In [ ]:
import zipfile
from google.cloud import storage
import os
import pandas as pd
import datetime as dt
import geopandas as gpd
import shapely
import pyproj
import matplotlib.pyplot as plt
from OSGridConverter import grid2latlong

## WWTW - TO CHECK

In [ ]:
determinands = pd.read_csv(
    "gs://rdmai_dev_data/raw/Continuous_Discharge/Determinands.csv"
)

In [ ]:
c_active = pd.read_csv(
    "gs://rdmai_dev_data/raw/Continuous_Discharge/consents_active.csv",
    encoding="latin-1"
)

In [ ]:
wwtw = c_active[c_active["DISCHARGE_SITE_TYPE_CODE"] == 'A2']

In [ ]:
wwtw.columns, determinands.columns

In [ ]:
dete_codes = [7782, 7729, 135, 85, 111, 6051, 348, 6057, 3647, 8174, 6425]

In [ ]:
determinands = determinands[determinands["DETE_CODE"].isin(dete_codes)]

In [ ]:
# Get code values together

determinands['CODE_VALUE'] = determinands.apply(
    lambda x: [
        (x[f'CODE_{i}'], x[f'VAL_{i}'])
        for i in range(1, 4)
        if pd.notna(x[f'CODE_{i}']) and pd.notna(x[f'VAL_{i}'])
    ],
    axis=1
)
determinands = determinands.explode('CODE_VALUE')
determinands[['CODE', 'VALUE']] = pd.DataFrame(
    determinands['CODE_VALUE'].tolist(), index=determinands.index)
determinands = determinands.drop(['CODE_VALUE', 'CODE_1', 'CODE_2', 'CODE_3',
                                  'VAL_1', 'VAL_2', 'VAL_3'], axis=1)

In [ ]:
determinands

In [ ]:
pivoted_determinands = pd.pivot_table(determinands, values="VALUE", index=[
                                      "PERMIT_REF", "VERSION", "OUTLET_NUMBER", "EFFLUENT_NUMBER", "CODE"], columns=["DETE"])

In [ ]:
wwtw_ren = wwtw.rename(
    columns={"PERMIT_NUMBER": "PERMIT_REF", "PERMIT_VERSION": "VERSION"})
wwtw_ren.set_index(
    ["PERMIT_REF", "VERSION", "OUTLET_NUMBER", "EFFLUENT_NUMBER"])

In [ ]:
wwtw_cleaned = wwtw_ren.set_index(["PERMIT_REF", "VERSION", "OUTLET_NUMBER", "EFFLUENT_NUMBER"]).join(
    pivoted_determinands, how='left')

In [ ]:
gdf = gpd.GeoDataFrame(wwtw_cleaned)

In [ ]:
gdf

In [ ]:
gdf["OUTLET_LOC"] = gdf["OUTLET_GRID_REF"].apply(
    lambda x: shapely.Point(grid2latlong(x).longitude, grid2latlong(x).latitude))
gdf["EFFLUENT_LOC"] = gdf["EFFLUENT_GRID_REF"].apply(
    lambda x: shapely.Point(grid2latlong(x).longitude, grid2latlong(x).latitude))
gdf["DISCHARGE_LOC"] = gdf["DISCHARGE_NGR"].apply(
    lambda x: shapely.Point(grid2latlong(x).longitude, grid2latlong(x).latitude))

In [ ]:
gdf["EFFLUENT_LOC"] = gdf["EFFLUENT_LOC"].astype('geometry')
gdf["OUTLET_LOC"] = gdf["OUTLET_LOC"].astype('geometry')
gdf["DISCHARGE_LOC"] = gdf["DISCHARGE_LOC"].astype('geometry')

In [ ]:
gdf = gdf.set_geometry("DISCHARGE_LOC").set_crs(epsg=4326)

In [ ]:
gdf = gdf.reset_index()

In [ ]:
gdf.shape

In [ ]:
gdf.to_parquet("gs://rdmai_dev_data/cleansed/Theme2CD/WWTW.parquet")

## SSSIs

In [ ]:
bucketName = "gs://rdmai_dev_data/raw/CSO_Historical_Data/NWG/".split("/")[2]
path_in_bucket = '/'.join(
    "gs://rdmai_dev_data/raw/CSO_Historical_Data/NWG/".split("/")[3:])
storage_client = storage.Client()
bucket = storage_client.get_bucket(bucketName)
geojson_filenames = ["gs://"+bucketName+'/' +
                     fname.name for fname in bucket.list_blobs(prefix=path_in_bucket) if fname.name.endswith('.geojson')]
cso = gpd.read_file(geojson_filenames[-1])
cso_all = cso.to_crs(epsg=3857)

In [ ]:
gdf = gpd.read_file(
    "gs://rdmai_dev_data/raw/Sites_of_Special_Scientific_Interest/Sites_of_Special_Scientific_Interest_England.zip")

In [ ]:
gdf = gdf.to_crs(epsg=3857)

In [ ]:
gdf.to_file("gs://rdmai_dev_data/cleansed/Theme2CD/SSSI.geojson")

In [ ]:
gpd.read_file("gs://rdmai_dev_data/cleansed/Theme2CD/SSSI.geojson")

## Bathing Waters

In [ ]:
df = pd.read_csv("gs://rdmai_dev_data/raw/Bathing_Water_Data.csv")

In [ ]:
gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(
    df["long"], df["lat"]), crs="EPSG:4326")

In [ ]:
gdf.columns

In [ ]:
gdf.to_file("gs://rdmai_dev_data/cleansed/Theme2CD/BathingWaters.geojson")

## Consents

In [ ]:
def get_lat_lon(x):
    try:
        latlong = grid2latlong(x)
        return shapely.Point(latlong.longitude, latlong.latitude)
    except:
        return None

In [ ]:
gdf = pd.read_csv(
    "gs://rdmai_dev_data/raw/Continuous_Discharge/consents_active.csv",
    encoding="latin-1"
)

In [ ]:
gdf["OUTLET_LOC"] = gdf["OUTLET_GRID_REF"].apply(
    get_lat_lon).astype('geometry')
gdf["EFFLUENT_LOC"] = gdf["EFFLUENT_GRID_REF"].apply(
    get_lat_lon).astype('geometry')
gdf["DISCHARGE_LOC"] = gdf["DISCHARGE_NGR"].apply(
    get_lat_lon).astype('geometry')

In [ ]:
gdf = gdf.set_geometry("OUTLET_LOC").set_crs(epsg=4326).to_crs(epsg=3857)
gdf = gdf.set_geometry("EFFLUENT_LOC").set_crs(epsg=4326).to_crs(epsg=3857)
gdf = gdf.set_geometry("DISCHARGE_LOC").set_crs(epsg=4326).to_crs(epsg=3857)

### WWTW

In [ ]:
rec_env_codes = [
    1,  # Freshwater River
    6,  # Canal
    # 7,  # Pond/Lake/Reservoir
    9,  # Estuary
    15,  # Into land then to watercourse
]

In [ ]:
gdf = gdf[
    (gdf["DISCHARGE_SITE_TYPE_CODE"] == "A2") &
    (gdf["EFFLUENT_TYPE"] == "SA") &
    (gdf["OUTLET_TYPE_CODE"] == "S") &
    (gdf["RECEIVING_ENVIRON_TYPE_CODE"].isin(rec_env_codes))
]

In [ ]:
gdf = gdf.drop_duplicates(
    subset=["PERMIT_NUMBER", "OUTLET_GRID_REF"], keep='first')

In [ ]:
gdf.shape

In [ ]:
gdf.to_parquet("gs://rdmai_dev_data/cleansed/Theme2CD/WWTW_consents.parquet")

In [ ]:
gdf = gpd.read_parquet(
    "gs://rdmai_dev_data/cleansed/Theme2CD/WWTW_consents.parquet")

### CSOs

In [ ]:
gdf.columns

In [ ]:
rec_env_codes = [
    1,  # Freshwater River
    6,  # Canal
    # 7,  # Pond/Lake/Reservoir
    9,  # Estuary
    15,  # Into land then to watercourse
]
effluent_types = ["SB", "SC", "SD"]

In [ ]:
gdf = gdf[
    (gdf["OUTLET_TYPE_CODE"] == "S") &
    (gdf["RECEIVING_ENVIRON_TYPE_CODE"].isin(rec_env_codes)) &
    (gdf["EFFLUENT_TYPE"].isin(effluent_types))
]

In [ ]:
gdf = gdf.drop_duplicates(
    subset=["PERMIT_NUMBER", "OUTLET_GRID_REF"], keep='first')

In [ ]:
gdf.to_parquet("gs://rdmai_dev_data/cleansed/Theme2CD/CSO_consents.parquet")

In [ ]:
gdf = gpd.read_parquet(
    "gs://rdmai_dev_data/cleansed/Theme2CD/CSO_consents.parquet")

In [ ]:
import numpy as np
np.mean([])

## Determinands

In [ ]:
determinands = pd.read_csv(
    "gs://rdmai_dev_data/raw/Continuous_Discharge/Determinands.csv"
)

In [ ]:
dete_codes = [
    7782,                   # Flow : Dry Weather :- {DWF}
    7729,                   # Flow : Dry Weather : 3 in 5 Years
    135,                    # Solids, Suspended at 105 C
    85,                     # BOD : 5 Day ATU
    111,                    # Ammoniacal Nitrogen as N
    6051,                   # Iron
    348,                    # Phosphorus, Total as P
    6057,                   # Aluminium
    3647,                   # Flow : To full treatment (Before spill to stor...
    8174,                   # Weir Setting
    6425,  # Flow : Daily total
]

In [ ]:
determinands = determinands[determinands["DETE_CODE"].isin(dete_codes)]

In [ ]:
# Get code values together

determinands['CODE_VALUE'] = determinands.apply(
    lambda x: [
        (x[f'CODE_{i}'], x[f'VAL_{i}'])
        for i in range(1, 4)
        if pd.notna(x[f'CODE_{i}']) and pd.notna(x[f'VAL_{i}'])
    ],
    axis=1
)
determinands = determinands.explode('CODE_VALUE')
determinands[['CODE', 'VALUE']] = pd.DataFrame(
    determinands['CODE_VALUE'].tolist(), index=determinands.index)
determinands = determinands.drop(['CODE_VALUE', 'CODE_1', 'CODE_2', 'CODE_3',
                                  'VAL_1', 'VAL_2', 'VAL_3'], axis=1)

In [ ]:
determinands = determinands.reset_index(drop=True)

In [ ]:
determinands.to_parquet(
    "gs://rdmai_dev_data/cleansed/Theme2CD/determinands.parquet")

## CSOs - ArcGIS

In [ ]:
csos = gpd.read_file(
    "https://services-eu1.arcgis.com/MSNNjkZ51iVh8yBj/arcgis/rest/services/Pledge2_view/FeatureServer/0/query?where=1%3D1&outFields=*&f=geojson&outSR=3857")  # NWL

## Risk calculation

### CSOs

In [ ]:
csos = gpd.read_file(
    "gs://rdmai_dev_data/cleansed/Theme2CD/CSO_arcgis.geojson")
df = pd.read_excel(
    "gs://rdmai_dev_data/cleansed/Theme2CD/CSO_Annual_2023.xlsx", header=1, sheet_name=None, index_col=None)
annual = pd.concat(df.values())

In [ ]:
consents = gpd.read_parquet(
    "gs://rdmai_dev_data/cleansed/Theme2CD/CSO_consents.parquet")

In [ ]:
joined = csos.join(annual.set_index("Unique ID")[
    "EA Permit Reference\n(EA Consents Database)"], on="SITE_ID", how="left").rename(columns={"EA Permit Reference\n(EA Consents Database)": "EA_Permit"})
joined

In [ ]:
joined2 = joined.join(consents.set_index("PERMIT_NUMBER")[
                      ["PERMIT_VERSION", "OUTLET_NUMBER", "EFFLUENT_NUMBER", "DISCHARGE_LOC"]], on="EA_Permit", how='left')
joined2

In [ ]:
determinands = pd.read_parquet(
    "gs://rdmai_dev_data/cleansed/Theme2CD/determinands.parquet")
determinands.DETE.unique()

In [ ]:
joined2["MATCH_ID"] = joined2.apply(lambda x: "_".join(
    [str(x["EA_Permit"]),
     str(x["PERMIT_VERSION"]),
     str(x["OUTLET_NUMBER"]),
     str(x["EFFLUENT_NUMBER"])
     ]), axis=1)
determinands["MATCH_ID"] = determinands.apply(lambda x: "_".join([
    str(x["PERMIT_REF"]),
    str(x["VERSION"]),
    str(x["OUTLET_NUMBER"]),
    str(x["EFFLUENT_NUMBER"])
]), axis=1)

In [ ]:
determinands[determinands["MATCH_ID"].isin(joined2["MATCH_ID"])]

In [ ]:
determinands[
    (determinands["DETE"].str.contains("Phosphorus"))
]

In [ ]:
joined3 = joined2.join(determinands[
    (determinands["DETE"].str.contains("Weir"))
    # & (determinands["CODE"].str.contains("95"))
].set_index("MATCH_ID")["VALUE"], on="MATCH_ID", how='left')
joined3

In [ ]:
joined3.VALUE.count()

In [ ]:
joined3[(joined3["SITE_ID"].duplicated(False)) & (
    ~joined3["MATCH_ID"].str.contains("nan"))].sort_values("MATCH_ID")

### WwTW

In [ ]:
wwtw = gpd.read_parquet(
    "gs://rdmai_dev_data/cleansed/Theme2CD/WWTW_consents.parquet")
determinands = pd.read_parquet(
    "gs://rdmai_dev_data/cleansed/Theme2CD/determinands.parquet")

#### Phosphorus

In [ ]:
joined = wwtw.join(determinands[determinands["DETE"].str.contains("Phosphorus") & (
    determinands["CODE"] == "MEAN VALUE")].set_index("ID")["VALUE"], on="ID", how="left")
joined.rename(columns={"VALUE": "P"}, inplace=True)

In [ ]:
joined["P"].fillna(5, inplace=True)

#### DWF

In [ ]:
joined = joined.join(determinands[(determinands["DETE"].str.contains("Flow : Dry Weather :- {DWF}")) & (
    determinands["CODE"] == "MAXIMUM VALUE")].set_index("ID")["VALUE"], on="ID", how="left")
joined.rename(columns={"VALUE": "DWF"}, inplace=True)

In [ ]:
joined["DWF"].fillna(25, inplace=True)

## Testing

In [ ]:
determinands

In [ ]:
determinands.set_index("ID", inplace=True)

In [ ]:
determinands[determinands["CODE"] ==
             "MAXIMUM VALUE"].loc["AWCNF10454_1_1_1", "VALUE"]

In [ ]:
determinands.to_parquet(
    "gs://rdmai_dev_data/cleansed/Theme2CD/determinands.parquet")